# IEP-2001 — 하이브리드 검색 실험 (BM25 + Vector)

> **목표**: 벡터 단독 검색과 하이브리드 검색(BM25 + Vector RRF)의 RAGAS 4지표 비교  
> **Baseline**: IEP-1001 CASE 3 벡터 단독 (Recall 0.8537, Precision 0.6117, Faithfulness 0.2748, AR 0.3409)  
> **작성일**: 2026-04-27  
> **환경**: Google Colab T4

---

## 실행 순서

```
Cell 1   패키지 설치
Cell 2   경로 설정 (본인 Drive 경로 수정 필요 ⚠️)
Cell 3   ChromaDB 로드 + 전체 청크 추출
Cell 4   BM25 인덱스 구축
Cell 5   검색 함수 정의
Cell 6   단일 질문 비교 테스트 ← 눈으로 먼저 확인
Cell 7   Threshold 처리 방식 확인
Cell 8   골든 데이터셋 100개 하이브리드 검색 실행
Cell 9   Solar 답변 생성 (100개)
Cell 10  RAGAS Context Recall + Precision 측정 (llama judge)
Cell 11  RAGAS Faithfulness 측정 (Solar judge)
Cell 12  RAGAS Answer Relevancy 측정 (Solar judge + jhgan)
Cell 13  결과 정리 + 벡터 단독 비교표 출력
Cell 14  Drive 저장
```

---
## Cell 1 — 패키지 설치

In [ ]:
# numpy 먼저 고정 (Colab 바이너리 충돌 방지)
import subprocess
subprocess.run(["pip", "uninstall", "numpy", "-y"], capture_output=True)
subprocess.run(["pip", "install", "numpy==1.26.4", "--force-reinstall", "-q"], capture_output=True)
print("numpy 재설치 완료 → 런타임 재시작 필요 없음 (처음 실행 시에만)")

# BM25 + 한국어 토크나이저
!pip install rank-bm25 kiwipiepy --quiet

# RAG 스택
!pip install chromadb==0.5.11 langchain-community langchain-huggingface \
             sentence-transformers openai python-dotenv ragas --quiet

import numpy as np
print(f"numpy: {np.__version__}")  # 1.26.4 확인

---
## Cell 2 — Google Drive 마운트 + 경로 설정

> ⚠️ **본인 Drive 경로에 맞게 수정하세요**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── ⚠️ 본인 경로로 수정 ──────────────────────────────────────────────
BASE_DIR        = "/content/drive/MyDrive/IPCC_data"          # 데이터 루트
CHROMA_DIR      = f"{BASE_DIR}/IEP_1001_CASE3/chroma_db"                     # ChromaDB 경로
GOLDEN_PATH     = f"{BASE_DIR}/IEP_1002/golden_dataset_100.csv"           # 골든 데이터셋
RESULTS_DIR     = f"{BASE_DIR}/IEP_2001/results_hybrid"                # 결과 저장 폴더
# ─────────────────────────────────────────────────────────────────────

COLLECTION_NAME = "ipcc_1001_case3_v1"
EMBED_MODEL     = "jhgan/ko-sroberta-multitask"

# 결과 폴더 생성
os.makedirs(RESULTS_DIR, exist_ok=True)

# 경로 확인
print(f"ChromaDB 존재: {os.path.exists(CHROMA_DIR)}")
print(f"골든 데이터셋 존재: {os.path.exists(GOLDEN_PATH)}")
print(f"결과 폴더: {RESULTS_DIR}")

In [ ]:
import getpass, os

# Solar API 키

# Colab Secrets(🔑)에 등록한 경우 아래 주석 해제
# from google.colab import userdata
# UPSTAGE_API_KEY = userdata.get('UPSTAGE_API_KEY')

UPSTAGE_API_KEY = getpass.getpass("UPSTAGE_API_KEY 입력: ")
os.environ["UPSTAGE_API_KEY"] = UPSTAGE_API_KEY
print(f"✅ API 키 설정 완료 (앞 8자리: {UPSTAGE_API_KEY[:8]}...)")

---
## Cell 3 — ChromaDB 로드 + 전체 청크 추출

BM25 인덱스 구축을 위해 ChromaDB에서 전체 청크 텍스트를 가져옵니다.

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

# 임베딩 모델 로드
print("임베딩 모델 로드 중...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

# ChromaDB 로드
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

# 전체 청크 추출 (BM25 인덱스 구축용)
raw = vectorstore._collection.get(include=["documents", "metadatas"])
docs_text = raw["documents"]    # List[str]
docs_meta = raw["metadatas"]    # List[dict]

count = vectorstore._collection.count()
print(f"\nChromaDB 로드 완료: {count}개 청크")
print(f"docs_text 길이: {len(docs_text)}")
print(f"메타데이터 샘플: {docs_meta[0]}")
# ✅ 확인: 506개, metadata에 'page'와 'source' 키 있어야 함

---
## Cell 4 — BM25 인덱스 구축

kiwipiepy(형태소 분석) → 실패 시 공백 split으로 fallback

In [ ]:
from rank_bm25 import BM25Okapi
import time

# 토크나이저 설정
try:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    def tokenize(text: str):
        return [token.form for token in kiwi.tokenize(text)]
    tokenizer_name = "kiwipiepy (형태소 분석)"
except Exception as e:
    print(f"kiwipiepy 로드 실패: {e}")
    def tokenize(text: str):
        return text.split()
    tokenizer_name = "공백 split (fallback)"

print(f"토크나이저: {tokenizer_name}")

# 전체 코퍼스 토크나이징 + BM25 인덱스 구축
print(f"\n{len(docs_text)}개 청크 토크나이징 중...")
t0 = time.time()
tokenized_corpus = [tokenize(doc) for doc in docs_text]
bm25 = BM25Okapi(tokenized_corpus)
elapsed = time.time() - t0

print(f"BM25 인덱스 구축 완료: {elapsed:.1f}초")
print(f"샘플 토큰 (청크 0): {tokenized_corpus[0][:15]}")

---
## Cell 5 — 검색 함수 정의

### 하이브리드 검색 구조
```
질문
 ├─ 벡터 검색  → (doc, cosine_score) × BM25_CANDIDATE_K개
 │              → SIMILARITY_THRESHOLD 필터 (기존 거절 로직 유지)
 │
 └─ BM25 검색  → (doc, bm25_score)   × BM25_CANDIDATE_K개
                       ↓
              RRF 합산: score = Σ 1 / (RRF_K + rank)
                       ↓
              TOP_K개 반환
```

**Threshold 처리 (방법 C 채택)**  
벡터 검색에서 threshold 미달 → 빈 리스트 반환 → 거절 응답 (기존 로직과 동일)

In [ ]:
import os
from langchain_core.documents import Document

# ── 파라미터 ────────────────────────────────────────────────────────
TOP_K                = 3      # 최종 반환 청크 수
BM25_CANDIDATE_K     = 10     # BM25/벡터 각각에서 뽑는 후보 수
SIMILARITY_THRESHOLD = 0.28   # 벡터 score 임계값 (2026-04-22 확정)
RRF_K                = 60     # RRF 하이퍼파라미터 (표준값)
# ────────────────────────────────────────────────────────────────────


def vector_search(question: str, k: int = BM25_CANDIDATE_K):
    """벡터 검색 → List[(Document, cosine_score)]"""
    return vectorstore.similarity_search_with_relevance_scores(question, k=k)


def bm25_search(question: str, k: int = BM25_CANDIDATE_K):
    """BM25 검색 → List[(Document, bm25_score)]"""
    tokens = tokenize(question)
    scores = bm25.get_scores(tokens)
    top_indices = scores.argsort()[::-1][:k]
    results = []
    for idx in top_indices:
        doc = Document(
            page_content=docs_text[idx],
            metadata=docs_meta[idx]
        )
        results.append((doc, float(scores[idx])))
    return results


def rrf_fusion(vector_results, bm25_results, top_k: int = TOP_K):
    """
    Reciprocal Rank Fusion
    rrf_score = Σ 1 / (RRF_K + rank)  (rank: 0-indexed)
    두 리스트의 순위를 합산해 top_k개 반환
    """
    rrf_scores = {}   # doc_key → rrf_score 누적
    doc_map    = {}   # doc_key → Document

    def doc_key(doc: Document) -> str:
        # page_content 앞 80자를 키로 사용 (중복 청크 방지)
        return doc.page_content[:80]

    for rank, (doc, _) in enumerate(vector_results):
        k_ = doc_key(doc)
        rrf_scores[k_] = rrf_scores.get(k_, 0.0) + 1.0 / (RRF_K + rank + 1)
        doc_map[k_] = doc

    for rank, (doc, _) in enumerate(bm25_results):
        k_ = doc_key(doc)
        rrf_scores[k_] = rrf_scores.get(k_, 0.0) + 1.0 / (RRF_K + rank + 1)
        doc_map[k_] = doc

    sorted_keys = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
    return [(doc_map[k_], rrf_scores[k_]) for k_ in sorted_keys[:top_k]]


def hybrid_search(question: str):
    """
    하이브리드 검색 메인 함수 (방법 C: 벡터 threshold 선행 필터)
    - 벡터 검색 결과 중 SIMILARITY_THRESHOLD 미달 시 빈 리스트 반환 → 거절
    - threshold 통과한 벡터 결과 + BM25 결과 → RRF → TOP_K 반환
    """
    # 1. 벡터 검색
    v_results = vector_search(question, k=BM25_CANDIDATE_K)

    # 2. threshold 필터 (기존 거절 로직 유지)
    v_filtered = [(doc, s) for doc, s in v_results if s >= SIMILARITY_THRESHOLD]
    if not v_filtered:
        return []   # 범위 밖 질문 → 거절

    # 3. BM25 검색
    b_results = bm25_search(question, k=BM25_CANDIDATE_K)

    # 4. RRF 합산
    return rrf_fusion(v_filtered, b_results, top_k=TOP_K)


print("검색 함수 정의 완료")
print(f"  TOP_K={TOP_K}, CANDIDATE_K={BM25_CANDIDATE_K}, THRESHOLD={SIMILARITY_THRESHOLD}, RRF_K={RRF_K}")

---
## Cell 6 — 단일 질문 비교 테스트

RAGAS 측정 전에 눈으로 먼저 확인합니다.  

**확인 포인트**:
- 하이브리드가 벡터 단독과 다른 청크를 가져오는가?
- `1.5°C` 같은 수치/고유명사 질문에서 BM25가 더 관련 있는 청크를 올려오는가?
- 경계 케이스(`환경 보호 개인 실천`, score 0.2624)에서 거절이 유지되는가?

In [ ]:
test_questions = [
    "1.5°C 목표를 달성하기 위해 필요한 탄소 감축량은?",      # 수치/고유명사 → BM25 유리 예상
    "기후변화가 생태계에 미치는 영향은?",                     # 의미적 질문 → Vector 유리 예상
    "환경 보호를 위해 개인이 할 수 있는 일은?",               # 경계 케이스 (score 0.2624)
    "오늘 점심 메뉴 추천해줘",                               # 완전 범위 밖 → 거절 확인
]

for q in test_questions:
    print(f"\n{'='*65}")
    print(f"질문: {q}")

    # 벡터 단독
    v_results = vector_search(q, k=TOP_K)
    print(f"\n[벡터 단독] scores: {[round(s, 4) for _, s in v_results]}")
    for doc, score in v_results:
        page = doc.metadata.get('page', '?')
        print(f"  score={score:.4f} | page={page} | {doc.page_content[:70].strip()}...")

    # 하이브리드
    h_results = hybrid_search(q)
    if not h_results:
        print(f"\n[하이브리드] → 거절 (threshold 미달)")
    else:
        print(f"\n[하이브리드] rrf_scores: {[round(s, 4) for _, s in h_results]}")
        for doc, score in h_results:
            page = doc.metadata.get('page', '?')
            print(f"  rrf={score:.4f}  | page={page} | {doc.page_content[:70].strip()}...")

    # 차이 분석
    if h_results:
        v_pages = {doc.metadata.get('page') for doc, _ in v_results}
        h_pages = {doc.metadata.get('page') for doc, _ in h_results}
        new_pages = h_pages - v_pages
        if new_pages:
            print(f"  → 하이브리드에서 새로 등장한 page: {new_pages}")
        else:
            print(f"  → 벡터 단독과 동일한 청크")

---
## Cell 7 — 골든 데이터셋 로드 + 100개 하이브리드 검색 실행

In [ ]:
# 파일 형식 확인 (임시 셀)
with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    content = f.read()

print(f"파일 크기: {len(content)} chars")
print(f"앞 200자:\n{content[:200]}")
print(f"\n뒤 100자:\n{content[-100:]}")

In [ ]:
import csv

with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    golden = list(reader)

print(f"골든 데이터셋: {len(golden)}개")
print(f"키 확인: {list(golden[0].keys())}")
print(f"샘플 question: {golden[0]['user_input'][:60]}")
print(f"샘플 reference: {golden[0]['reference'][:60]}")

In [ ]:
import time

print("100개 하이브리드 검색 실행 중...")
t0 = time.time()

hybrid_dataset = []
rejected_count = 0

for i, item in enumerate(golden):
    q            = item["user_input"]       # ← question → user_input
    ground_truth = item["reference"]        # ← ground_truth → reference

    results  = hybrid_search(q)
    contexts = [doc.page_content for doc, _ in results]

    if not contexts:
        rejected_count += 1

    hybrid_dataset.append({
        "question":        q,
        "ground_truth":    ground_truth,
        "contexts":        contexts,
        "retrieved_count": len(contexts),
        "rejected":        len(contexts) == 0,
    })

    if (i + 1) % 20 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/100 완료 ({elapsed:.0f}초)")

elapsed = time.time() - t0
avg_chunks = sum(r["retrieved_count"] for r in hybrid_dataset) / len(hybrid_dataset)

print(f"\n완료: {elapsed:.1f}초")
print(f"거절 수: {rejected_count}/100")
print(f"평균 검색 청크 수: {avg_chunks:.2f}")

---
## Cell 8 — Solar 답변 생성 (100개)

기존 `IEP1001_case3_ragas_solar.ipynb`와 동일한 프롬프트 사용

In [ ]:
from openai import OpenAI
import time

SOLAR_BASE_URL = "https://api.upstage.ai/v1"
SOLAR_MODEL    = "solar-pro3"

solar_client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url=SOLAR_BASE_URL
)

RAG_PROMPT = """당신은 IPCC AR6 기후변화 보고서 전문 안내 AI입니다.
반드시 아래 제공된 문서 내용만을 근거로 답변하세요.
문서에 없는 내용은 추측하지 말고 "해당 내용은 IPCC 보고서에서 찾을 수 없습니다."라고 답하세요.

[참고 문서]
{context}

[질문]
{question}

[답변]"""

REJECTION_MESSAGE = (
    "죄송합니다. 해당 질문은 IPCC AR6 보고서의 범위를 벗어납니다. "
    "IPCC 기후변화 관련 질문을 입력해 주세요."
)

def generate_answer(question: str, contexts: list) -> str:
    if not contexts:
        return REJECTION_MESSAGE
    context_text = "\n\n".join(
        f"[청크 {i+1}]\n{c}" for i, c in enumerate(contexts)
    )
    prompt = RAG_PROMPT.format(context=context_text, question=question)
    resp = solar_client.chat.completions.create(
        model=SOLAR_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=800
    )
    return resp.choices[0].message.content


print("Solar 답변 생성 중 (100개)...")
t0 = time.time()

for i, item in enumerate(hybrid_dataset):
    item["answer"] = generate_answer(item["question"], item["contexts"])
    if (i + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/100 완료 ({elapsed:.0f}초)")

total = time.time() - t0
print(f"\n답변 생성 완료: {total:.0f}초")

---
## Cell 9 — RAGAS 설정 (공통)

Solar RAGAS 호환 설정 (IEP1001_case3_ragas_solar.ipynb 동일)

In [ ]:
import pandas as pd
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from ragas import EvaluationDataset
from langchain_huggingface import HuggingFaceEmbeddings

# ── IEP-1001 기준 수치 (비교용) ──────────────────────────────────
IEP1001_RESULTS = {
    "전체":     {"context_recall": 0.8537, "context_precision": 0.6117, "faithfulness": 0.2748, "answer_relevancy": 0.3409},
    "사실 확인": {"context_recall": 0.7381, "context_precision": 0.9133, "faithfulness": 0.5125, "answer_relevancy": 0.5910},
    "비교":     {"context_recall": 0.8600, "context_precision": 0.6733, "faithfulness": 0.2818, "answer_relevancy": 0.2670},
    "의견/예측": {"context_recall": 0.9757, "context_precision": 0.7800, "faithfulness": 0.2619, "answer_relevancy": 0.4342},
    "범위 밖":  {"context_recall": 0.8264, "context_precision": 0.0800, "faithfulness": 0.0774, "answer_relevancy": 0.0714},
}

# ── Solar judge LLM ──────────────────────────────────────────────
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=SOLAR_MODEL,
        openai_api_key=UPSTAGE_API_KEY,
        openai_api_base=SOLAR_BASE_URL,
        temperature=0,
        max_tokens=512,
    )
)

# ── jhgan 임베딩 (Solar embedding $.input 오류로 대체) ────────────
ragas_embeddings_hf = LangchainEmbeddingsWrapper(embeddings)  # Cell 3에서 로드한 객체 재사용

# ── RunConfig: Solar 타임아웃 대응 ───────────────────────────────
run_config = RunConfig(max_workers=2, timeout=120, max_retries=3)

# ── EvaluationDataset 구성 ────────────────────────────────────────
# golden 데이터셋의 Type 컬럼 활용 (유형별 집계용)
eval_records = []
for item, h in zip(golden, hybrid_dataset):
    eval_records.append({
        "id":                 item.get("ID", ""),
        "type":               item.get("Type", ""),
        "user_input":         h["question"],
        "retrieved_contexts": h["contexts"],
        "response":           h["answer"],
        "reference":          h["ground_truth"],
    })

eval_dataset = EvaluationDataset.from_pandas(pd.DataFrame([{
    "user_input":         r["user_input"],
    "retrieved_contexts": r["retrieved_contexts"],
    "response":           r["response"],
    "reference":          r["reference"],
} for r in eval_records]))

print(f"RAGAS 평가 설정 완료")
print(f"  Judge LLM  : {SOLAR_MODEL}")
print(f"  Embeddings : jhgan/ko-sroberta-multitask (AR용)")
print(f"  RunConfig  : max_workers=2, timeout=120, max_retries=3")
print(f"  batch_size : 1 (Solar 타임아웃 방지)")
print(f"  EvaluationDataset: {len(eval_dataset)}개")
print(f"  유형 분포: {pd.Series([r['type'] for r in eval_records]).value_counts().to_dict()}")


---
## Cell 10 — RAGAS Context Recall + Precision 측정

judge: llama3.1:8b (또는 Solar 대체)  
IEP-1001 Baseline: Recall 0.8537 / Precision 0.6117

In [ ]:
from ragas import evaluate
from ragas.metrics import ContextRecall, ContextPrecision
import time

# llama 연결 실패 → Solar로 대체
# ⚠️ IEP-1001(llama 기준 0.8537/0.6117)과 직접 비교 불가 — README 주석 필수
print("Context Recall + Precision 측정 중 (Solar judge — llama 연결 실패로 대체)...")
t0 = time.time()

result_retrieval = evaluate(
    dataset=eval_dataset,
    metrics=[ContextRecall(), ContextPrecision()],
    llm=ragas_llm,          # llm_llama → ragas_llm
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)

elapsed = time.time() - t0
df_retrieval = result_retrieval.to_pandas()
df_retrieval["id"]   = [r["id"]   for r in eval_records]
df_retrieval["type"] = [r["type"] for r in eval_records]

recall_mean    = df_retrieval["context_recall"].mean(skipna=True)
precision_mean = df_retrieval["context_precision"].mean(skipna=True)
recall_nan     = df_retrieval["context_recall"].isna().sum()
precision_nan  = df_retrieval["context_precision"].isna().sum()

print(f"\n완료: {elapsed:.0f}초")
print(f"  Context Recall    : {recall_mean:.4f}  (NaN: {recall_nan}개)")
print(f"  Context Precision : {precision_mean:.4f}  (NaN: {precision_nan}개)")
print(f"\n⚠️  judge: Solar (llama 연결 실패). IEP-1001 llama 기준 수치와 직접 비교 불가.")
print(f"     IEP-1001 참고값: Recall 0.8537 / Precision 0.6117 (llama judge)")

---
## Cell 11 — RAGAS Faithfulness 측정

judge: Solar  
IEP-1001 Baseline: 0.2748 (NaN 38개, 보수적 0.1704)

In [ ]:
from ragas.metrics import Faithfulness
import time

SAVE_FAITHFULNESS = f"{RESULTS_DIR}/iep2001_faithfulness.csv"

print(f"Faithfulness 측정 중 (Solar judge, batch_size=1)...")
t0 = time.time()

result_faith = evaluate(
    dataset=eval_dataset,
    metrics=[Faithfulness()],
    llm=ragas_llm,
    embeddings=ragas_embeddings_hf,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)

elapsed = time.time() - t0
df_faith = result_faith.to_pandas()
df_faith["id"]   = [r["id"]   for r in eval_records]
df_faith["type"] = [r["type"] for r in eval_records]
df_faith.to_csv(SAVE_FAITHFULNESS, index=False, encoding="utf-8-sig")

faith_mean = df_faith["faithfulness"].mean(skipna=True)
faith_nan  = df_faith["faithfulness"].isna().sum()
faith_conservative = round(faith_mean * (100 - faith_nan) / 100, 4)

print(f"\n완료: {elapsed:.0f}초")
print(f"  Faithfulness      : {faith_mean:.4f}  (NaN: {faith_nan}개)")
print(f"  보수적 수치        : {faith_conservative:.4f}  (NaN→0, 전체 100개 기준)")
print(f"  IEP-1001 기준     : {IEP1001_RESULTS['전체']['faithfulness']:.4f}  diff: {faith_mean - IEP1001_RESULTS['전체']['faithfulness']:+.4f}")
print(f"  저장: {SAVE_FAITHFULNESS}")


---
## Cell 12 — RAGAS Answer Relevancy 측정

judge: Solar + jhgan 임베딩 + strictness=1  
IEP-1001 Baseline: 0.3409 (Solar n=1 제약)

In [ ]:
from ragas.metrics import AnswerRelevancy
import time

SAVE_RELEVANCY = f"{RESULTS_DIR}/iep2001_relevancy.csv"

# Solar n=1 제약 → strictness=1 필수 (기본값 3 사용 시 전량 실패)
metric_ar = AnswerRelevancy(strictness=1)

print("Answer Relevancy 측정 중 (Solar judge + jhgan + strictness=1, batch_size=1)...")
t0 = time.time()

result_ar = evaluate(
    dataset=eval_dataset,
    metrics=[metric_ar],
    llm=ragas_llm,
    embeddings=ragas_embeddings_hf,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)

elapsed = time.time() - t0
df_ar = result_ar.to_pandas()
df_ar["id"]   = [r["id"]   for r in eval_records]
df_ar["type"] = [r["type"] for r in eval_records]
df_ar.to_csv(SAVE_RELEVANCY, index=False, encoding="utf-8-sig")

ar_mean = df_ar["answer_relevancy"].mean(skipna=True)
ar_nan  = df_ar["answer_relevancy"].isna().sum()

print(f"\n완료: {elapsed:.0f}초")
print(f"  Answer Relevancy  : {ar_mean:.4f}  (NaN: {ar_nan}개)")
print(f"  IEP-1001 기준     : {IEP1001_RESULTS['전체']['answer_relevancy']:.4f}  diff: {ar_mean - IEP1001_RESULTS['전체']['answer_relevancy']:+.4f}")
print(f"  저장: {SAVE_RELEVANCY}")


---
## Cell 13 — 결과 정리 + 벡터 단독 비교표

In [ ]:
import numpy as np
from datetime import datetime

METRICS = ["faithfulness", "answer_relevancy"]
TYPES   = ["사실 확인", "비교", "의견/예측", "범위 밖"]

# ── 전체 raw 합산 ────────────────────────────────────────────────
df_raw = pd.DataFrame({
    "id":               [r["id"]   for r in eval_records],
    "type":             [r["type"] for r in eval_records],
    "user_input":       [r["user_input"] for r in eval_records],
    "context_recall":   df_retrieval["context_recall"].values,
    "context_precision":df_retrieval["context_precision"].values,
    "faithfulness":     df_faith["faithfulness"].values,
    "answer_relevancy": df_ar["answer_relevancy"].values,
})

ALL_METRICS = ["context_recall", "context_precision", "faithfulness", "answer_relevancy"]
overall_mean = df_raw[ALL_METRICS].mean(skipna=True).round(4)
overall_nan  = df_raw[ALL_METRICS].isna().sum()
summary      = df_raw.groupby("type")[ALL_METRICS].mean().round(4)

# ── 생존 편향 보정 ────────────────────────────────────────────────
faith_conservative = round(overall_mean["faithfulness"] * (100 - overall_nan["faithfulness"]) / 100, 4)
ar_conservative    = round(overall_mean["answer_relevancy"] * (100 - overall_nan["answer_relevancy"]) / 100, 4)

# ── 비교표 출력 ──────────────────────────────────────────────────
print("=" * 70)
print("  IEP-2001 vs IEP-1001 비교 (Solar judge 기준, 동일 조건)")
print("=" * 70)
comparison = pd.DataFrame([
    {
        "단계":               "벡터 단독 (IEP-1001)",
        "Recall":             IEP1001_RESULTS["전체"]["context_recall"],
        "Precision":          IEP1001_RESULTS["전체"]["context_precision"],
        "Faithfulness":       IEP1001_RESULTS["전체"]["faithfulness"],
        "Faith(보수적)":     0.1704,
        "Answer Relevancy":  IEP1001_RESULTS["전체"]["answer_relevancy"],
    },
    {
        "단계":               "하이브리드 (IEP-2001)",
        "Recall":             overall_mean["context_recall"],
        "Precision":          overall_mean["context_precision"],
        "Faithfulness":       overall_mean["faithfulness"],
        "Faith(보수적)":     faith_conservative,
        "Answer Relevancy":  overall_mean["answer_relevancy"],
    },
])
print(comparison.to_string(index=False))

print(f"\n거절 수: {rejected_count}/100")

# ── 유형별 비교 ──────────────────────────────────────────────────
print("\n[유형별 상세]")
for m in ALL_METRICS:
    print(f"\n  {m}")
    print(f"    {'유형':<10} {'IEP-2001':>10} {'IEP-1001':>10} {'차이':>8}")
    print(f"    {'-'*42}")
    for t in TYPES:
        v2 = summary.loc[t, m] if t in summary.index else float("nan")
        v1 = IEP1001_RESULTS.get(t, {}).get(m, float("nan"))
        diff = f"{v2-v1:+.4f}" if not (np.isnan(v2) or np.isnan(v1)) else "   N/A"
        print(f"    {t:<10} {v2:>10.4f} {v1:>10.4f} {diff:>8}")
    print(f"    {'-'*42}")
    print(f"    {'전체':<10} {overall_mean[m]:>10.4f} {IEP1001_RESULTS['전체'][m]:>10.4f} {overall_mean[m]-IEP1001_RESULTS['전체'][m]:>+8.4f}")

# ── 생존 편향 보정 ────────────────────────────────────────────────
print("\n[생존 편향 보정]")
print(f"  {'지표':<22} {'낙관적(NaN제외)':>16} {'유효샘플':>8} {'보수적(전체100개)':>18}")
print(f"  {'-'*68}")
for m, pes in [("faithfulness", faith_conservative), ("answer_relevancy", ar_conservative)]:
    opt = overall_mean[m]
    vn  = 100 - overall_nan[m]
    print(f"  {m:<22} {opt:>16.4f} {vn:>8}개 {pes:>18.4f}")

# ── README 마크다운 출력 ─────────────────────────────────────────
print("\n[README 마크다운]")
md  = "| 유형 | Context Recall | Context Precision | Faithfulness | Answer Relevancy |\n"
md += "| :--- | :---: | :---: | :---: | :---: |\n"
for t in TYPES:
    cr = summary.loc[t, "context_recall"]    if t in summary.index else float("nan")
    cp = summary.loc[t, "context_precision"] if t in summary.index else float("nan")
    fa = summary.loc[t, "faithfulness"]      if t in summary.index else float("nan")
    ar = summary.loc[t, "answer_relevancy"]  if t in summary.index else float("nan")
    md += f"| {t} | {cr:.4f} | {cp:.4f} | {fa:.4f} | {ar:.4f} |\n"
md += (f"| **전체** | **{overall_mean['context_recall']:.4f}** "
       f"| **{overall_mean['context_precision']:.4f}** "
       f"| **{overall_mean['faithfulness']:.4f}** "
       f"| **{overall_mean['answer_relevancy']:.4f}** |\n")
print(md)

# ── Drive 저장 ───────────────────────────────────────────────────
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
SAVE_RAW     = f"{RESULTS_DIR}/iep2001_raw_{timestamp}.csv"
SAVE_SUMMARY = f"{RESULTS_DIR}/iep2001_summary_{timestamp}.csv"

df_raw.to_csv(SAVE_RAW, index=False, encoding="utf-8-sig")
df_s = summary.copy()
df_s.loc["전체"] = overall_mean
df_s.to_csv(SAVE_SUMMARY, encoding="utf-8-sig")

print(f"저장 완료")
print(f"  raw     → {SAVE_RAW}")
print(f"  summary → {SAVE_SUMMARY}")
print(f"\n⚠️  주석")
print(f"  Faithfulness: Solar judge, NaN {overall_nan['faithfulness']}개 포함 — 보수적 수치 {faith_conservative}")
print(f"  Answer Relevancy: strictness=1 (Solar n=1 제약), jhgan 임베딩")
if not use_llama:
    print(f"  ⚠️  llama judge 미사용 → Recall/Precision도 Solar judge 기준. IEP-1001과 직접 비교 주의.")


---
## Cell 14 — Drive 저장

In [ ]:
import json
import pandas as pd
import os
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# 1. summary CSV
summary_path = os.path.join(RESULTS_DIR, f"iep2001_summary_{timestamp}.csv")
pd.DataFrame([results_summary]).to_csv(summary_path, index=False, encoding="utf-8-sig")
print(f"summary 저장: {summary_path}")

# 2. 개별 결과 CSV (RAGAS 원본)
df_all = df_retrieval.copy()
df_all["faithfulness"]     = df_faith["faithfulness"]
df_all["answer_relevancy"] = df_ar["answer_relevancy"]
raw_path = os.path.join(RESULTS_DIR, f"iep2001_raw_{timestamp}.csv")
df_all.to_csv(raw_path, index=False, encoding="utf-8-sig")
print(f"raw 결과 저장: {raw_path}")

# 3. 하이브리드 검색 결과 (contexts 포함)
dataset_path = os.path.join(RESULTS_DIR, f"iep2001_dataset_{timestamp}.json")
with open(dataset_path, "w", encoding="utf-8") as f:
    json.dump(hybrid_dataset, f, ensure_ascii=False, indent=2)
print(f"dataset 저장: {dataset_path}")

print("\n✅ 모든 파일 저장 완료")
print(f"저장 위치: {RESULTS_DIR}")